# M2 — Category

Predicts the case `category` (10 classes) from the **summary only**, on a **group-aware split**. Adds a *confidence analysis* (how much more confident the model is when it is right), which can gate low-confidence suggestions.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
search = os.getcwd()
while search != os.path.dirname(search) and not os.path.exists(os.path.join(search, "case_pipeline.py")):
    search = os.path.dirname(search)
sys.path.insert(0, search)
import numpy as np, pandas as pd
import case_pipeline as P   # shared loading / splits / features / evaluation

In [ ]:
df      = P.load_clean()
FEATURES = ["case_summary"]        # summary only; tags would leak the category
TARGET   = "category"
classes  = sorted(df[TARGET].unique())

train, test = P.group_split(df)
model = P.make_clf([P.t_word(), P.t_char()]).fit(train[FEATURES], train[TARGET])
pred  = model.predict(test[FEATURES])
proba = model.predict_proba(test[FEATURES])
confidence = proba.max(axis=1)

baseline = df[TARGET].value_counts(normalize=True).iloc[0]
metrics  = P.evaluate(test[TARGET], pred, classes, baseline=baseline)

# Signature metric: mean confidence gap between correct and incorrect predictions.
correct = (pred == test[TARGET].values)
metrics["confidence_gap"] = round(float(confidence[correct].mean() - confidence[~correct].mean()), 3)
metrics

In [ ]:
heldout = test[["case_id", "case_summary", TARGET]].copy().reset_index(drop=True)
heldout["predicted"]  = pred
heldout["confidence"] = confidence.round(3)

deployed = P.make_clf([P.t_word(), P.t_char()]).fit(df[FEATURES], df[TARGET])
P.save_bundle("m2_category", {
    "model": deployed, "target": TARGET, "features": FEATURES, "classes": classes, "kind": "multiclass",
    "label": "Category", "metrics": metrics, "test": heldout,
    "report": P.class_report(test[TARGET], pred, classes),
    "confusion": P.conf_matrix(test[TARGET], pred, classes),
    "meta": {"contributor": "B", "approach": "summary-only text classifier",
             "split": "group-on-summary", "inputs": "summary only (tags excluded — leakage)"}})